# Module 22 — Asyncio

## Exercise 22.1 — Twelve asyncio predictions

Predict the OUTPUT ORDER and the TOTAL TIME for each before running.
REQUIRES PYTHON 3.11+ for asyncio.TaskGroup (q07), except* (q07) and
asyncio.timeout (q10). On 3.10 those three are skipped with a message; the
other nine run unchanged. Check with:  python --version
Run:  python ex01_basics.py

---

**How to work through this.** Each task below is its own cell. Run them one at a
time and read the output before moving on; that is the whole advantage of a
notebook over a script. Where a cell asks for a prediction, write it before you
run anything. Being wrong on purpose in a place where it costs nothing is how
the correct model gets built.

---

# The concepts behind this exercise

Read this before the tasks. Every idea the tasks below use is explained here, so
you should not need to leave this notebook.

The code cells in this part are demonstrations rather than exercises. Run them,
change a value, run them again. That is the whole point of having them here
instead of in a document.

## Concept 1. The event loop

One thread, one loop, a queue of ready tasks.

```text
loop:
    run each READY task until it awaits something
    ask the OS which of the awaited things are now ready   (epoll/kqueue)
    move those tasks back to READY
    repeat
```


There is no preemption. A task runs until **it** yields control by awaiting. That
single fact explains both asyncio's biggest advantage — no locks needed between
awaits, because nothing can interleave — and its biggest failure mode, below.

In [ ]:
import asyncio

async def main() -> None:
    await asyncio.sleep(1)
    print("done")

asyncio.run(main())          # creates a loop, runs main, closes the loop

---

## Concept 2. Coroutines, awaitables, tasks

```text
async def fetch(url: str) -> str: ...

coro = fetch(url)            # NOTHING has run yet -- exactly like a generator
result = await coro          # runs it, suspending here until it completes
task = asyncio.create_task(coro)     # schedules it to run CONCURRENTLY
result = await task                   # waits for it
```


**Calling a coroutine function runs nothing.** It returns a coroutine object.
This is Module 14's "calling a generator function runs nothing", and forgetting
it produces the most common asyncio bug:

```text
fetch(url)                   # RuntimeWarning: coroutine was never awaited
await fetch(url)             # correct
```


**`await` does not create concurrency.** It waits. Concurrency comes from having
several tasks scheduled:

```text
# SEQUENTIAL: 3 seconds
a = await fetch(url1)
b = await fetch(url2)
c = await fetch(url3)

# CONCURRENT: 1 second
a, b, c = await asyncio.gather(fetch(url1), fetch(url2), fetch(url3))
```


That distinction is the second most common bug: code that is fully `async` and
entirely sequential.

---

## Concept 5. Cancellation and timeouts

```text
async with asyncio.timeout(5.0):        # 3.11+
    await slow_operation()               # raises TimeoutError

result = await asyncio.wait_for(slow_operation(), timeout=5.0)
```


Cancellation works by **raising `CancelledError` inside the coroutine at its
current `await`**. Two consequences:

```text
try:
    await something()
except asyncio.CancelledError:
    await cleanup()          # cleanup is fine
    raise                    # but you MUST re-raise
```


Swallowing `CancelledError` makes a task uncancellable, and `TaskGroup` and
timeouts stop working for it. Note it inherits from `BaseException` (not
`Exception`) since 3.8, precisely so `except Exception` does not eat it —
the same design reasoning as `KeyboardInterrupt` in Module 16.

**Cancellation only happens at an `await`.** A coroutine in a tight
non-awaiting loop cannot be cancelled at all.

---

## Concept 8. Choosing: async, threads, or processes

| | asyncio | threads | processes |
|---|---|---|---|
| Concurrency limit | ~100,000 | ~hundreds | ~CPU count |
| Memory per unit | a few KB | ~8 MB stack | a full interpreter |
| CPU parallelism | no | no (pure Python) | **yes** |
| Blocking calls | **poison the loop** | fine | fine |
| Ecosystem | needs async libraries | any library | any library |
| Debugging | harder | hard | hardest |
| Rewrite cost | **whole call stack** | none | none |

**The async colour problem is the real cost.** An `async` function can only be
awaited by another `async` function, so making one function async makes its
entire call chain async. Introducing asyncio into an existing sync codebase is
not a local change.

**Rule of thumb:** hundreds of concurrent I/O operations, or a framework that is
already async (FastAPI) → asyncio. A few dozen blocking calls in otherwise sync
code → threads. CPU work → processes.

---

---

# Now the exercise

You have everything you need. Work top to bottom, and where a cell asks for a
prediction, write it before you run anything.

## The concepts this exercise uses

These are the numbered sections of [the module README](../README.md). If a task below stops making sense, the section named next to it is the one to re-read.

- Section 1: The event loop
- Section 2: Coroutines, awaitables, tasks
- Section 3: `TaskGroup` over `gather`
- Section 4: The number one asyncio bug: blocking the loop
- Section 5: Cancellation and timeouts
- Section 6: Async iteration and context managers
- Section 7: Bridging sync and async
- Section 8: Choosing: async, threads, or processes

> The teaching for this module currently lives in the README rather than in this notebook. Read it alongside these cells.

## Setup

Run this first. It is the imports and any shared values the tasks below need.

In [ ]:
from __future__ import annotations

import asyncio
import sys
import time

HAS_311 = sys.version_info >= (3, 11)

---

## `work`

_work_

In [ ]:
async def work(name: str, seconds: float) -> str:
    print(f"      start {name}")
    await asyncio.sleep(seconds)
    print(f"      end   {name}")
    return name

---

## `q01`

_q01_

In [ ]:
async def q01() -> None:
    # PREDICTION: order? total time?
    await work("a", 0.2)
    await work("b", 0.2)

---

## `q02`

_q02_

In [ ]:
async def q02() -> None:
    # PREDICTION:
    await asyncio.gather(work("a", 0.2), work("b", 0.2))

---

## `q03`

_q03_

In [ ]:
async def q03() -> None:
    # PREDICTION: what does this print, and what warning appears?
    work("never-awaited", 0.1)
    await asyncio.sleep(0.05)

---

## `q04`

_q04_

In [ ]:
async def q04() -> None:
    # PREDICTION: does creating the task start it? When does it run?
    task = asyncio.create_task(work("task", 0.1))
    print("      created the task")
    await asyncio.sleep(0)          # what does sleep(0) do?
    print("      after sleep(0)")
    await task

---

## `q05`

_q05_

In [ ]:
async def q05() -> None:
    # PREDICTION: total time? Which finishes first?
    results = await asyncio.gather(work("slow", 0.3), work("fast", 0.1))
    print("      results:", results)

---

## `q06`

_q06_

In [ ]:
async def q06() -> None:
    # PREDICTION: what happens to "b" when "a" raises?
    async def failing() -> None:
        await asyncio.sleep(0.05)
        raise ValueError("a failed")

    try:
        await asyncio.gather(failing(), work("b", 0.3))
    except ValueError as exc:
        print(f"      caught {exc}; is b still running?")
    await asyncio.sleep(0.4)

---

## `q07`

_q07_

In [ ]:
async def q07() -> None:
    # PREDICTION: same, but with TaskGroup. What is different?
    if not HAS_311:
        print("      SKIPPED: TaskGroup and except* need Python 3.11+")
        return
    # The 3.11+ body lives in a separate file so this module still imports on
    # 3.10 -- `except*` is a SYNTAX error, not a runtime one, so a version
    # check cannot guard it in the same file. That is worth noticing: syntax
    # introduced by a new version cannot be feature-detected inline.
    from _tg_311 import q07_taskgroup      # type: ignore[import-not-found]
    await q07_taskgroup(work)

---

## `q08`

_q08_

In [ ]:
async def q08() -> None:
    # PREDICTION: THE BIG ONE. What is the total time, and why?
    async def blocking() -> None:
        print("      blocking start")
        time.sleep(0.3)              # NOT asyncio.sleep
        print("      blocking end")

    start = time.perf_counter()
    await asyncio.gather(blocking(), work("a", 0.1), work("b", 0.1))
    print(f"      total {time.perf_counter() - start:.2f}s")

---

## `q09`

_q09_

In [ ]:
async def q09() -> None:
    # PREDICTION: and with to_thread?
    def blocking() -> None:
        time.sleep(0.3)

    start = time.perf_counter()
    await asyncio.gather(asyncio.to_thread(blocking), work("a", 0.1))
    print(f"      total {time.perf_counter() - start:.2f}s")

---

## `q10`

_q10_

In [ ]:
async def q10() -> None:
    # PREDICTION: does the cleanup run? Does the timeout fire?
    async def stubborn() -> None:
        try:
            await asyncio.sleep(10)
        except asyncio.CancelledError:
            print("      stubborn: caught cancellation, cleaning up")
            # note: NOT re-raised
    if not HAS_311:
        print("      partial: asyncio.timeout needs 3.11+; using wait_for")
        try:
            await asyncio.wait_for(stubborn(), timeout=0.1)
            print("      wait_for returned -- the timeout did NOT cancel it")
        except asyncio.TimeoutError:
            print("      TimeoutError raised")
        return
    try:
        async with asyncio.timeout(0.1):
            await stubborn()
        print("      timeout did NOT fire")
    except TimeoutError:
        print("      TimeoutError raised")

---

## `q11`

_q11_

In [ ]:
async def q11() -> None:
    # PREDICTION: single-threaded. Can this race?
    counter = 0

    async def increment() -> None:
        nonlocal counter
        for _ in range(1000):
            current = counter
            await asyncio.sleep(0)      # a suspension point
            counter = current + 1

    await asyncio.gather(*(increment() for _ in range(5)))
    print(f"      counter = {counter} (expected 5000)")

---

## `q12`

_q12_

In [ ]:
async def q12() -> None:
    # PREDICTION: what does the semaphore change about the timing?
    sem = asyncio.Semaphore(2)

    async def limited(name: str) -> None:
        async with sem:
            await work(name, 0.1)

    start = time.perf_counter()
    await asyncio.gather(*(limited(f"t{i}") for i in range(6)))
    print(f"      total {time.perf_counter() - start:.2f}s for 6 tasks, limit 2")

---

## `main`

_main_

In [ ]:
async def main() -> None:
    for fn in [q01, q02, q03, q04, q05, q06, q07, q08, q09, q10, q11, q12]:
        print(f"\n{fn.__name__}")
        start = time.perf_counter()
        await fn()
        print(f"    ({time.perf_counter() - start:.2f}s)")

---

## Run it

This is what running the original file did. Everything above must have been run first.

In [ ]:
if __name__ == "__main__":
    asyncio.run(main())

---

## Before you move on

- [ ] Every cell above ran, in order, on a fresh kernel.
- [ ] You wrote a prediction before running, wherever one was asked for.
- [ ] You can say in one sentence what each task was actually testing.
- [ ] Anything that surprised you is written down in `PROGRESS.md`.

Compare against the worked answers in `../solutions/` only after your own
attempt runs.